# Initialize DAQ, Chip and Periphery

In [1]:
from tjmonopix2.system.bdaq53 import BDAQ53
from tjmonopix2.system.tjmonopix2 import TJMonoPix2
import numpy as np
import time
# from tjmonopix2.system.periphery import Periphery

bdaq = BDAQ53()
bdaq.init()

chip = TJMonoPix2(daq=bdaq, chip_sn="W14R13")
chip.init()

# bdaq.configure_ptdc_module()
# bdaq.enable_ptdc_module()

2025-06-17 16:22:29,979 [TJ-Monopix2      ] - SUCCESS Found board BDAQ53 running firmware version 0.1
2025-06-17 16:22:30,127 [basil.HL.si570   ] - INFO    Changed Si570 reference frequency to 160.0 MHz
2025-06-17 16:22:30,330 [TJ-Monopix2 - W14R13] - WARNING No explicit configuration supplied. Using 'default.cfg.yaml'!


In [2]:

# Enable readout

chip.registers['CMOS_TX_EN_CONF'].write(1)
bdaq.set_LEMO_MUX('LEMO_MUX_TX0', 1)  # CMD_LOOP_START_PULSE
# bdaq["tdc"].EN_INVERT_TRIGGER = 0
chip.registers["SEL_PULSE_EXT_CONF"].write(0)

delay = 40
rd_frz_dly = 40
chip.registers["READ_START_CONF"].write(1 + delay + rd_frz_dly)
chip.registers["READ_STOP_CONF"].write(5 + delay + rd_frz_dly)
chip.registers["FREEZE_START_CONF"].write(1 + delay)
chip.registers["FREEZE_STOP_CONF"].write(40 + delay + rd_frz_dly)
chip.registers["LOAD_CONF"].write(39 + delay + rd_frz_dly)
chip.registers["STOP_CONF"].write(40 + delay + rd_frz_dly)

col, row = 2, 2
# Enable / TDAC
chip.masks.reset_all()

chip.masks['enable'][col, row] = True
chip.masks['tdac'][col, row] = 0b100
chip.masks['injection'][col, row] = True
chip.masks['hitor'][col, row] = True

time.sleep(0.1)

chip.registers['ICASN'].write(5)
chip.registers['IBIAS'].write(100)
chip.registers['VRESET'].write(115)

time.sleep(0.1)
_ = chip.masks.update(force=True)
time.sleep(0.1)

2025-06-17 16:22:44,167 [TJ-Monopix2      ] - INFO    LEMO_MUX_TX0 set to 1 (CMD_LOOP_START_PULSE)


In [3]:
bdaq.enable_ptdc_module()
bdaq.configure_ptdc_module()
# time.sleep(0.1)
# bdaq.calibrate_ptdc_module()
# # data = bdaq["FIFO"].get_data()
# # print(data)

# time.sleep(1)
# bdaq.reset_fifo()
# time.sleep(0.1)

bdaq.rx_channels["rx0"].set_en(True)
# Start CMD LOOP PULSER, 160 MHz
bdaq['pulser_cmd_start_loop'].set_en(True)
bdaq['pulser_cmd_start_loop'].set_width(8)
bdaq['pulser_cmd_start_loop'].set_delay(246)
bdaq['pulser_cmd_start_loop'].set_repeat(1)


# bdaq.enable_ptdc_module()
chip.registers["VH"].write(41)
chip.registers["VL"].write(1)
time.sleep(0.05)
chip.inject(PulseStartCnfg=19, PulseStopCnfg=19 + 1024, wait_cycles=1, latency=700, repetitions=5000)
time.sleep(.01)
data = bdaq["FIFO"].get_data()
print(len(data), data)
hit, reg = chip.interpret_data(data)
bdaq.disable_ptdc_module()

2025-06-17 16:22:57,945 [TJ-Monopix2      ] - INFO    Configuring TDC module


20001 [1812016375 1275068432 1255978026 ... 1270295108 1190134320 1105200508]


In [4]:
print(data[data & 0xF0000000 == 0x60000000])
data[data & 0xF0000000 == (0x6 << 28)]
data[data & 0xF0000000 == (0x4 << 28)]

[1812016375]


array([1275068432, 1255978026, 1190134320, ..., 1270295108, 1190134320,
       1105200508], dtype=uint32)

In [5]:
print(hit['col'], len(hit['col']))
print(hit[hit['col'] > 512])

[2 2 2 ... 2 2 2] 5000
[]


In [6]:
for i in range(20):
    test = bdaq['pTDC'].disassemble_tdc_word(data[i])
    print(test)

{'source_id': np.uint32(6), 'word_type': 'RST', 'timestamp': np.uint32(150), 'raw_word': np.uint32(1812016375)}
{'source_id': np.uint32(4), 'word_type': 'RST', 'timestamp': np.uint32(0), 'raw_word': np.uint32(1275068432)}
{'source_id': np.uint32(4), 'word_type': 'MISS', 'raw_word': np.uint32(1255978026)}
{'source_id': np.uint32(4), 'word_type': 'TIMESTAMP', 'timestamp': np.uint32(30721), 'raw_word': np.uint32(1190134320)}
{'source_id': np.uint32(4), 'word_type': 'TRIGGERED', 'tdl_value': np.uint32(124), 'fine_clk_value': np.uint32(190472), 'raw_word': np.uint32(1106249084)}
{'source_id': np.uint32(4), 'word_type': 'RST', 'timestamp': np.uint32(0), 'raw_word': np.uint32(1275068432)}
{'source_id': np.uint32(4), 'word_type': 'MISS', 'raw_word': np.uint32(1255980888)}
{'source_id': np.uint32(4), 'word_type': 'TIMESTAMP', 'timestamp': np.uint32(30721), 'raw_word': np.uint32(1190134320)}
{'source_id': np.uint32(4), 'word_type': 'TRIGGERED', 'tdl_value': np.uint32(124), 'fine_clk_value': np.u

In [ ]:
# bdaq.configure_ptdc_module()
# bdaq.calibrate_ptdc_module()
# print(ptdc)

# calib_data_indices = bdaq['pTDC'].is_calib_word(ptdc)
# if any(calib_data_indices) :
#     calib_values = bdaq['pTDC'].get_raw_tdl_values(np.array(ptdc[calib_data_indices]))
#     print(calib_values)
#     bdaq['pTDC'].set_calib_values(calib_values)
#     bdaq.log.info("Calibration set using %s samples" % len(calib_values))
    
# print(bdaq['pTDC'].calib_vector)

In [7]:

# plt.plot((bdaq['pTDC'].calib_vector))
# plt.vlines(96, 0, 1.2, color='r', label='96th bin')
# plt.xlabel('Calibration bin')
# plt.ylabel('Calibration vector value')
# plt.grid()
# plt.legend()

In [8]:

# bdaq.reset_fifo()
# bdaq.enable_ptdc_module()
# chip.registers["VH"].write(31)
# chip.registers["VL"].write(1)

# time.sleep(0.05)
# chip.inject(PulseStartCnfg=19, PulseStopCnfg=19 + 1024, wait_cycles=1, latency=700, repetitions=10000)
# time.sleep(.01)
# data = bdaq["FIFO"].get_data()
# # print(len(data), data)
# hit, reg = chip.interpret_data(data)
# bdaq.disable_ptdc_module()

In [9]:
# print(hit['col'])

In [10]:
# for i in range(len(data)):
#     test = bdaq['pTDC'].disassemble_tdc_word(data[i])
#     if test["word_type"] != "TRIGGERED":
#         print(test)

# for i in range(20):
#     test = bdaq['pTDC'].disassemble_tdc_word(data[i])
#     print(test)

In [11]:
# time_word_indices = bdaq['pTDC'].is_time_word(data)
# time_data = data[time_word_indices]
# if any(time_word_indices) :
#         time_in_ns = bdaq['pTDC'].tdc_word_to_time(time_data)




# time_in_ns

In [12]:
# bdaq['pTDC'].EN_CALIBRATION_MODE = 1

# collected_data = np.empty(0, dtype=np.uint32)
# fifo_data = bdaq['FIFO'].get_data()
# data_size = len(fifo_data)
# collected_data = np.concatenate((collected_data, fifo_data), dtype=np.uint32)

# bdaq['pTDC'].EN_CALIBRATION_MODE = 0

# for i in range(len(collected_data)):
#     test = bdaq['pTDC'].disassemble_tdc_word(collected_data[i])
#     if test["word_type"] != "TRIGGERED":
#         print(test)

In [13]:
# from matplotlib import pyplot as plt
# plt.hist(np.diff(time_in_ns))
# plt.show()
# plt.hist(time_in_ns)
# plt.show()

# print(hit[hit[:]["col"] > 512])

In [14]:
# from matplotlib import pyplot as plt

# plt.hist(time_in_ns)
# plt.show()

In [15]:

# print(np.count_nonzero(bdaq['pTDC'].is_calib_word(data)))
# print(np.count_nonzero(bdaq['pTDC'].is_time_word(data)))
# # print(bdaq['pTDC'].tdc_word_to_time(data[bdaq['pTDC'].is_time_word(data)]))

# time_word_indices = bdaq['pTDC'].is_time_word(data)

# time_data = data[time_word_indices]
# # time_in_ns = bdaq['pTDC'].tdc_word_to_time(time_data)
# if any(time_word_indices) :
#     data_values = bdaq['pTDC'].get_raw_tdl_values(np.array(data[time_word_indices]))
#     print(data_values)
#     time_in_ns = bdaq['pTDC'].tdc_word_to_time(data_values)
#     bdaq.log.info("Calibration set using %s samples" % len(data_values))
# bdaq.disable_ptdc_module()
# print(time_in_ns)

# print(bdaq['pTDC'].calib_vector)

# Analog timing crosscheck Standard Front-End

In [16]:
# # Inject DeltaV = VH - VL [DAC] into injection mask
# chip.masks.reset_all()
# chip.masks['enable'][250, 250] = True
# chip.masks['injection'][250, 250] = True
# chip.masks['hitor'][250, 250] = True
# chip.masks.update(force=True)
# chip.registers['ICASN'].write(5)
# chip.registers['IBIAS'].write(100)
# chip.registers['VRESET'].write(115)

In [17]:
# chip.registers['VL'].write(1)
# chip.registers['VH'].write(141)
# for i in range(2250):
#     _ = chip.inject(PulseStartCnfg=19, PulseStopCnfg=19 + 1024, wait_cycles=1, latency=700, repetitions=800)